# Day 8: Timestamp-Safe Feature Engineering and Leakage Checks

This notebook documents the Day 8 feature pipeline for the weather probability modeling project.

The model target remains:

`forecast_error = actual_high - forecast_high`

The model is learning the distribution of forecast error, not raw temperature. Every feature must be timestamp-safe: for a row at `prediction_time`, the feature can only use information available at or before that timestamp.

## What Day 8 Created

Day 8 added a reproducible feature-building and leakage-auditing pipeline.

Created files:

- `src/features.py`: reusable feature engineering functions.
- `src/leakage_checks.py`: leakage and timestamp-safety checks.
- `scripts/build_features.py`: one-command build script.
- `tests/test_day8_features.py`: focused tests for Day 8 invariants.
- `data/processed/modeling_rows_v1.csv`: final timestamp-safe modeling table.
- `outputs/day8_features/feature_columns.json`: model feature list and exclusions.
- `outputs/day8_features/leakage_check_report.md`: leakage audit report.
- `outputs/day8_features/feature_missingness_report.csv`: missingness summary.
- `outputs/day8_features/modeling_rows_v1_preview.csv`: random 20-row preview.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

# Make the notebook work whether it is launched from the repo root or notebooks/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs" / "day8_features"

MODELING_ROWS_PATH = PROCESSED_DIR / "modeling_rows_v1.csv"
FEATURE_COLUMNS_PATH = OUTPUTS_DIR / "feature_columns.json"
LEAKAGE_REPORT_PATH = OUTPUTS_DIR / "leakage_check_report.md"
MISSINGNESS_PATH = OUTPUTS_DIR / "feature_missingness_report.csv"
PREVIEW_PATH = OUTPUTS_DIR / "modeling_rows_v1_preview.csv"

print(REPO_ROOT)

## Step 1: Input Data Inspection

Day 8 starts from the Day 7 supervised table, now expanded to 24 hourly prediction rows per `date/location`, and joins timestamp-safe information from the cleaned hourly observed and hourly forecast tables.

The available inputs were:

- `data/processed/supervised_forecast_error_rows.csv`
- `data/processed/hourly_clean.csv`
- `data/processed/hourly_forecasts_clean.csv`
- `data/processed/daily_clean.csv`
- `data/processed/forecasts_clean.csv`

The implementation confirms actual column names before feature construction and gracefully skips features that cannot be built from the available columns.

In [ ]:
input_files = {
    "Day 7 supervised rows": PROCESSED_DIR / "supervised_forecast_error_rows.csv",
    "Hourly observed weather": PROCESSED_DIR / "hourly_clean.csv",
    "Hourly forecast weather": PROCESSED_DIR / "hourly_forecasts_clean.csv",
    "Daily actuals": PROCESSED_DIR / "daily_clean.csv",
    "Daily forecasts": PROCESSED_DIR / "forecasts_clean.csv",
}

for label, path in input_files.items():
    df_head = pd.read_csv(path, nrows=3)
    df_cols = pd.read_csv(path, nrows=0).columns.tolist()
    print(f"\n{label}")
    print(f"path: {path.relative_to(REPO_ROOT)}")
    print(f"columns: {df_cols}")
    display(df_head)

## Step 2: Feature Engineering Design

The feature builder lives in `src/features.py`.

Main functions:

- `load_inputs(...)`: loads Day 7 rows plus cleaned weather and forecast data, standardizes dates and timestamps, and records data limitations.
- `add_time_features(df)`: adds cyclic day-of-year and hour features, month, season, and `forecast_horizon_hours`.
- `add_observed_weather_features(rows, hourly)`: joins observed weather available at or before `prediction_time`.
- `add_forecast_relative_features(rows, hourly_forecasts)`: adds forecast-relative features using forecast valid timestamps and documents missing issue-time limitations.
- `add_sequential_context_features(rows, hourly, hourly_forecasts)`: adds cumulative same-day context features using only rows up to `prediction_time`.
- `add_solar_time_features(df)`: adds `minutes_until_typical_peak`.
- `add_forecast_update_features(rows, forecasts)`: attempts forecast revision features only when repeated forecast runs are available.
- `handle_missing_features(df)`: drops rows missing critical fields and writes the missingness report.
- `build_feature_matrix(...)`: orchestrates all Day 8 feature steps.
- `write_feature_columns(df, output_path)`: writes the model feature list and excluded columns.

`forecast_horizon_hours` and `minutes_until_typical_peak` use 3 PM local time on `target_date` as the default typical peak temperature time.

### Timestamp-Safe Observed Features

Observed weather features are built from `hourly_clean.csv`.

Safety rules:

- `current_temp` is the latest observed temperature at or before `prediction_time` for the same location.
- `max_temp_so_far` is the maximum observed temperature on `target_date` using observations with `timestamp <= prediction_time`.
- `temp_change_60m`, `temp_change_120m`, `temp_change_180m`, `temp_change_240m`, and `temp_change_300m` compare current temperature to prior observations.
- `temp_acceleration_60m` measures whether the latest 1-hour temperature change is speeding up or slowing down relative to the previous hour.
- `temp_change_60m_minus_3h_avg_rate` compares the latest 1-hour change to the average hourly change over the last 3 hours.
- Sequential context features track current temp relative to max so far, minutes/hour of max so far, cumulative observed-vs-forecast error, recent new highs, range so far, temperature curve area, and near-integer-boundary duration.
- `temp_change_30m` is skipped because the source data is hourly, and pretending 30-minute precision would create a fake signal.
- `current_temp_source_time` and `max_temp_so_far_source_time` are kept as audit metadata, not model features.

### Forecast-Relative Features

Forecast-relative features are built from `hourly_forecasts_clean.csv` and the Day 7 `forecast_high`.

Created features include:

- `forecast_high`
- `forecast_temp_current_hour`
- `current_temp_minus_forecast_temp`
- `forecast_max_so_far`
- `max_so_far_minus_forecast_max_so_far`

Important limitation: the forecast inputs have valid timestamps but no issue/run/reference timestamp. Because of that, the pipeline cannot prove which forecast run was available at each `prediction_time`. The leakage report records this as a `WARN`. Future-valid-window features such as next-3-hour cloud cover and precipitation probability are skipped because they require an issue-time-safe forecast run.

## Step 3: Load Day 8 Outputs

The following cells inspect the actual generated Day 8 artifacts.

In [ ]:
modeling_rows = pd.read_csv(MODELING_ROWS_PATH)

datetime_cols = [
    "target_date",
    "prediction_time",
    "prediction_timestamp",
    "current_temp_source_time",
    "max_temp_so_far_source_time",
    "forecast_temp_source_valid_time",
    "forecast_max_so_far_source_valid_time",
]
for col in datetime_cols:
    if col in modeling_rows.columns:
        modeling_rows[col] = pd.to_datetime(modeling_rows[col], errors="coerce")

with open(FEATURE_COLUMNS_PATH, encoding="utf-8") as f:
    feature_spec = json.load(f)

feature_columns = feature_spec["feature_columns"]
excluded_columns = feature_spec["excluded_columns"]

print(f"Rows: {len(modeling_rows):,}")
print(f"Columns: {len(modeling_rows.columns):,}")
print(f"Model feature columns: {len(feature_columns):,}")
print(f"Target date range: {modeling_rows['target_date'].min().date()} to {modeling_rows['target_date'].max().date()}")
print(f"Prediction time range: {modeling_rows['prediction_time'].min()} to {modeling_rows['prediction_time'].max()}")

In [ ]:
print("Feature columns")
display(pd.DataFrame({"feature": feature_columns}))

print("Excluded columns")
display(pd.DataFrame({"excluded_column": excluded_columns}))

## Step 4: Preview the Modeling Rows

The preview file contains 20 random rows generated by `scripts/build_features.py`.

In [ ]:
preview = pd.read_csv(PREVIEW_PATH)
display(preview.head(20))

## Step 5: Manual Timestamp-Safety Checks

A good manual check is to inspect one target date across all 24 prediction times. `current_temp` should move naturally, `max_temp_so_far` should be nondecreasing, and source timestamps should never be after `prediction_time`.

In [ ]:
important_cols = [
    "target_date",
    "location",
    "prediction_time",
    "actual_high",
    "forecast_high",
    "forecast_error",
    "current_temp",
    "current_temp_source_time",
    "max_temp_so_far",
    "max_temp_so_far_source_time",
    "forecast_temp_current_hour",
    "forecast_temp_source_valid_time",
    "forecast_max_so_far",
    "forecast_max_so_far_source_valid_time",
]

# Pick a date where the first hourly max is well below the final high.
candidate_rows = []
for (location, target_date), group in modeling_rows.groupby(["location", "target_date"]):
    group = group.sort_values("prediction_time")
    if len(group) >= 24:
        early_gap = group.iloc[0]["actual_high"] - group.iloc[0]["max_temp_so_far"]
        if early_gap > 0:
            candidate_rows.append((location, target_date, early_gap))

location, target_date, early_gap = sorted(candidate_rows, key=lambda item: item[2], reverse=True)[0]
same_day = modeling_rows[
    (modeling_rows["location"] == location)
    & (modeling_rows["target_date"] == target_date)
].sort_values("prediction_time")

print(f"Selected date: {location} {target_date.date()} with early gap {early_gap:.1f} degrees")
display(same_day[important_cols])

In [ ]:
ordered = modeling_rows.sort_values(["location", "target_date", "prediction_time"])

checks = {
    "rows_per_date_location_min": int(ordered.groupby(["location", "target_date"]).size().min()),
    "rows_per_date_location_max": int(ordered.groupby(["location", "target_date"]).size().max()),
    "max_so_far_decreases": int((ordered.groupby(["location", "target_date"])["max_temp_so_far"].diff().dropna() < -1e-9).sum()),
    "current_temp_source_after_prediction": int((modeling_rows["current_temp_source_time"] > modeling_rows["prediction_time"]).sum()),
    "max_temp_so_far_source_after_prediction": int((modeling_rows["max_temp_so_far_source_time"] > modeling_rows["prediction_time"]).sum()),
    "forecast_temp_valid_after_prediction": int((modeling_rows["forecast_temp_source_valid_time"] > modeling_rows["prediction_time"]).sum()),
    "forecast_max_valid_after_prediction": int((modeling_rows["forecast_max_so_far_source_valid_time"] > modeling_rows["prediction_time"]).sum()),
    "max_temp_so_far_gt_actual_high_plus_tolerance": int((modeling_rows["max_temp_so_far"] > modeling_rows["actual_high"] + 0.5).sum()),
    "forecast_error_in_features": "forecast_error" in feature_columns,
    "actual_high_in_features": "actual_high" in feature_columns,
    "all_null_features_in_spec": [col for col in feature_columns if modeling_rows[col].isna().all()],
}

checks

Expected result: all timestamp and leakage counts above should be zero, and the target/audit columns should not be in `feature_columns`.

## Step 6: Missingness Report

Critical fields are required and rows missing them are dropped. Non-critical missing values are left as `NaN` for now.

The all-null optional features remain visible in the modeling table and missingness report, but they are excluded from `feature_columns.json`.

In [ ]:
missingness = pd.read_csv(MISSINGNESS_PATH)
display(missingness.head(25))

## Step 7: Leakage Report

`src/leakage_checks.py` runs five checks:

1. Target leakage check.
2. Future timestamp check.
3. Max-so-far sanity check.
4. Chronological validity check.
5. Feature reproducibility check.

The overall status is `WARN`, not `FAIL`, because the forecast data lacks issue/run/reference timestamps. The observed-weather timestamp checks pass.

In [ ]:
from IPython.display import Markdown

Markdown(LEAKAGE_REPORT_PATH.read_text(encoding="utf-8"))

## Step 8: Reproduce the Build

From the repository root, the Day 8 outputs are reproduced with:

```bash
python scripts/build_features.py
```

The test suite is run with:

```bash
python -m pytest
```

Current build summary:

- Rows: `38,424`
- Feature columns: `39`
- Critical rows dropped: `0`
- Leakage checks: `WARN`, with `0` failed checks

In [ ]:
# Uncomment these lines to rebuild and retest from the notebook.
# %cd {REPO_ROOT}
# !python scripts/build_features.py
# !python -m pytest

## Day 8 Takeaways

- The final table keeps `forecast_error` as the target only.
- `actual_high` stays in the table for audit but is excluded from model features.
- Observed-weather features use source timestamps at or before `prediction_time`.
- `max_temp_so_far` is built cumulatively within each target date and never uses later observations.
- Forecast-relative features are limited by the source data's missing issue-time metadata.
- Optional unsupported features are skipped and documented instead of silently fabricated.

This sets up the project for empirical baselines and distributional models in the later days.